# REE Lambda Fitting — Part 1: Data Preparation

This notebook loads the raw LA-ICP-MS data, cleans it, and saves a
tidy `df_analysis.parquet` file for use by the fitting and plotting notebooks.

**Outputs**
- `df_analysis.parquet` — cleaned REE + metadata table, one row per sample

**Workflow**
1. Define physical constants (REE list, ionic radii, CI chondrite values)
2. Load the Excel spreadsheet
3. Handle below-detection-limit values and coerce numeric types
4. Drop rows with no REE data at all
5. Save to Parquet


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd


## 2. Constants

$$\text{CI-normalised ratio:}\quad (\mathrm{REE}_i)_N = \frac{\mathrm{REE}_i}{\mathrm{CI}_i}$$

$$\text{Log-space pattern used for fitting:}\quad y_i = \ln\left(\frac{\mathrm{REE}_i}{\mathrm{CI}_i}\right)$$


In [ ]:
REE = ["La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd", "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu"]
# The 14 lanthanides used for lambda fitting.
# Even an incomplete series can produce reliable results:
# e.g. https://www.nature.com/articles/s41598-025-89227-2

IONIC_RADII = np.array([
    1.160, 1.143, 1.126, 1.109, 1.079, 1.066,
    1.053, 1.040, 1.027, 1.015, 1.004, 0.994,
    0.985, 0.977
], dtype=float)
# Shannon ionic radii (Å, VIII-fold coordination) for La→Lu.
# Used as the independent variable rᵢ in the O'Neill (2016) polynomial basis.

CI = np.array([
    0.2472, 0.6308, 0.0950, 0.4793, 0.1542, 0.0592,
    0.2059, 0.0375, 0.2540, 0.0554, 0.1645, 0.0258,
    0.1684, 0.0251
], dtype=float)
# CI chondrite normalising values (ppm) for La→Lu (McDonough & Sun 1995).
# Dividing by CI removes the Oddo-Harkins sawtooth pattern so that
# the REE pattern reflects geochemical fractionation, not nuclear stability.


## 3. Load raw data

Edit the path below to switch datasets (e.g. Sudholz 2022 Gawler vs 2023 Kimberley).


In [ ]:
DATA_PATH = "../MES - Anastasia/Sudholz_2023.xlsx"   # Kimberley
# DATA_PATH = "../MES - Anastasia/Sudholz_2022.xlsx" # Gawler

df = pd.read_excel(DATA_PATH)
df = df.rename(columns={"SAMPLE NO. ": "SAMPLE NO."})  # strip trailing space

print(f"Loaded {len(df)} rows from {DATA_PATH}")
df.head()


## 4. Clean and subset

- **Below-detection-limit handling**: cells containing `"Below LOD"` are replaced with `NaN`.
- **Numeric coercion**: all REE and metadata columns are cast to `float`.
- **Empty-row removal**: rows with *all* REE values missing are dropped.


In [ ]:
META_COLS = ["SAMPLE NO.", "PROVINCE", "LOCATION", "REFERENCE",
             "Temp (Celc)", "GPa", "Depth (km)"]

df_analysis = df[META_COLS + REE].copy()

# Replace "Below LOD" strings with NaN, then coerce to float
for col in REE:
    m = df_analysis[col].astype(str).str.strip().eq("Below LOD")
    df_analysis.loc[m, col] = np.nan
    df_analysis[col] = pd.to_numeric(df_analysis[col], errors="coerce")

# Coerce numeric metadata columns
for col in ["Temp (Celc)", "GPa", "Depth (km)"]:
    if col in df_analysis.columns:
        df_analysis[col] = pd.to_numeric(df_analysis[col], errors="coerce")

# Drop rows with no REE data at all, reset index
df_analysis = df_analysis.dropna(how="all", subset=REE).reset_index(drop=True)

print("df_analysis shape:", df_analysis.shape)
df_analysis.head()


## 5. Quick sanity check

In [ ]:
print("REE completeness (non-NaN count per element):")
print(df_analysis[REE].notna().sum().to_string())
print()
print("Metadata completeness:")
print(df_analysis[["Temp (Celc)", "GPa", "Depth (km)"]].notna().sum().to_string())


## 6. Save output

In [ ]:
df_analysis.to_csv("df_analysis.csv", index=True)
print("Saved df_analysis.csv —", len(df_analysis), "rows")
